# Retrieval Bottleneck Experiment — Extended Version

Bu notebook, hakem yorumlarında belirtilen **tüm eksiklikleri** gidermek için genişletilmiş deneysel altyapıyı içerir.

### Hakem Eleştirilerine Karşılık Gelen Genişletmeler

| # | Hakem Yorumu | Bu Notebook'taki Çözüm |
|---|---|---|
| 1 | En az 2-3 farklı veri seti | SQuAD, Natural Questions, TriviaQA |
| 2 | En az 2-3 farklı ölçekte generator | Flan-T5-Small, Base, Large |
| 3 | Farklı retriever'lar ve top-k değerleri | BM25, SBERT, Hybrid + k=1,3,5,10 |
| 4 | Bootstrap CI / istatistiksel testler | Bootstrap %95 CI + Wilcoxon signed-rank test |
| 5 | Pratik framework önerisi | Retrieval Bottleneck Diagnostic Framework (RBDF) |

**Gereksinimler:** Google Colab Pro (GPU), ~4-6 saat çalışma süresi

---
## 0. Kurulum ve Bağımlılıklar

In [ ]:
!pip -q install transformers datasets sentence-transformers evaluate faiss-cpu \
    pandas matplotlib tqdm rank_bm25 scipy seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 115.6 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import random
import string
import warnings
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import evaluate
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi
from scipy import stats
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print("All imports successful.")

All imports successful.


---
## 1. Global Konfigürasyon

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================
SEED = 42
SAMPLE_SIZE = 1000
OUTPUT_DIR = "outputs_extended"

# --- Hakem Yorumu 2: Birden fazla generator ---
GENERATOR_CONFIGS = {
    "flan-t5-small": "google/flan-t5-small",
    "flan-t5-base": "google/flan-t5-base",
    "flan-t5-large": "google/flan-t5-large",
}

# --- Hakem Yorumu 3: Birden fazla retriever ve top-k ---
RETRIEVER_CONFIGS = {
    "bm25": {"type": "sparse"},
    "sbert": {"type": "dense", "model": "sentence-transformers/all-MiniLM-L6-v2"},
    "hybrid": {"type": "hybrid", "model": "sentence-transformers/all-MiniLM-L6-v2", "alpha": 0.5},
}

TOP_K_VALUES = [1, 3, 5, 10]

# --- Hakem Yorumu 1: Birden fazla veri seti ---
DATASET_CONFIGS = {
    "squad": {
        "name": "squad",
        "split": "validation",
        "question_key": "question",
        "context_key": "context",
        "answers_key": "answers",
    },
    "natural_questions": {
        "name": "nq_open",
        "split": "validation",
        "question_key": "question",
        "answers_key": "answer",  # NQ open has list of answer strings
    },
    "triviaqa": {
        "name": "trivia_qa",
        "subset": "rc.nocontext",
        "split": "validation",
        "question_key": "question",
        "answers_key": "answer",
    },
}

# --- Hakem Yorumu 4: Bootstrap CI ayarları ---
BOOTSTRAP_N_RESAMPLES = 1000
BOOTSTRAP_CI_LEVEL = 0.95

# Decoding settings (sabit)
MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 50
NUM_BEAMS = 2
WARMUP_RUNS = 5
EMBEDDING_BATCH_SIZE = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Generators: {list(GENERATOR_CONFIGS.keys())}")
print(f"Retrievers: {list(RETRIEVER_CONFIGS.keys())}")
print(f"Top-k values: {TOP_K_VALUES}")
print(f"Datasets: {list(DATASET_CONFIGS.keys())}")

Device: cuda
Generators: ['flan-t5-small', 'flan-t5-base', 'flan-t5-large']
Retrievers: ['bm25', 'sbert', 'hybrid']
Top-k values: [1, 3, 5, 10]
Datasets: ['squad', 'natural_questions', 'triviaqa']


---
## 2. Yardımcı Fonksiyonlar

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def normalize_text(text: str) -> str:
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = " ".join(text.split())
    return text

def any_answer_in_context(answers: List[str], context: str) -> bool:
    normalized_context = normalize_text(context)
    return any(normalize_text(a) in normalized_context for a in answers)

def build_prompt(context: str, question: str) -> str:
    return f"Context: {context}\nQuestion: {question}\nAnswer:"

set_seed(SEED)
print("Utility functions ready.")

Utility functions ready.


---


**SQuAD:** Doğrudan gold context içerir.

**Natural Questions & TriviaQA:** Açık alan soru-cevap veri setleri. Gold context olmadığı için Wikipedia'dan passage havuzu oluşturulacak.

In [ ]:
# Wikipedia passage havuzu (NQ ve TriviaQA için)
print("Loading Wikipedia passages for open-domain QA datasets...")
wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

WIKI_PASSAGE_POOL_SIZE = 50000

wiki_passages = []
for i, article in enumerate(wiki_dataset):
    text = article["text"]
    paragraphs = [p.strip() for p in text.split("\n\n") if len(p.strip().split()) > 30]
    wiki_passages.extend(paragraphs)
    if len(wiki_passages) >= WIKI_PASSAGE_POOL_SIZE:
        break

wiki_passages = wiki_passages[:WIKI_PASSAGE_POOL_SIZE]
print(f"Wikipedia passage pool: {len(wiki_passages)} passages")

Loading Wikipedia passages for open-domain QA datasets...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia passage pool: 50000 passages


In [ ]:
def load_and_prepare_dataset(dataset_key: str, sample_size: int = SAMPLE_SIZE):
    """
    Veri setini yükler ve standardize eder.
    Returns: List[dict] with keys: id, question, gold_context, gold_answers, passage_pool
    """
    config = DATASET_CONFIGS[dataset_key]
    instances = []

    if dataset_key == "squad":
        ds = load_dataset(config["name"], split=config["split"])
        ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))
        passage_pool = list(set(ds["context"]))

        for i, sample in enumerate(ds):
            instances.append({
                "id": sample["id"],
                "question": sample["question"],
                "gold_context": sample["context"],
                "gold_answers": list(sample["answers"]["text"]),
            })
        return instances, passage_pool

    elif dataset_key == "natural_questions":
        ds = load_dataset(config["name"], split=config["split"])
        ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))

        for i, sample in enumerate(ds):
            answers = sample["answer"]
            if isinstance(answers, str):
                answers = [answers]

            # NQ open: gold context yok, Wikipedia havuzundan cevap içeren passage'ı bul
            gold_ctx = _find_gold_context_from_pool(answers, wiki_passages)
            instances.append({
                "id": f"nq_{i}",
                "question": sample["question"],
                "gold_context": gold_ctx,
                "gold_answers": answers,
            })
        # NQ open-domain: passage_pool = wiki
        return instances, wiki_passages

    elif dataset_key == "triviaqa":
        ds = load_dataset(config["name"], config.get("subset", "rc.nocontext"),
                         split=config["split"])
        ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))

        for i, sample in enumerate(ds):
            answers = sample["answer"]["aliases"] + [sample["answer"]["value"]]
            answers = list(set(answers))

            gold_ctx = _find_gold_context_from_pool(answers, wiki_passages)
            instances.append({
                "id": f"tqa_{i}",
                "question": sample["question"],
                "gold_context": gold_ctx,
                "gold_answers": answers,
            })
        return instances, wiki_passages


def _find_gold_context_from_pool(answers, passages, max_search=5000):
    """
    Passage havuzunda cevap içeren ilk passage'ı bul.
    Bulamazsa en kısa cevabı içeren basit bir cümle döndür.
    """
    for p in passages[:max_search]:
        if any_answer_in_context(answers, p):
            return p
    # Fallback: cevabı doğrudan context olarak ver (bu instance'lar ayrı işaretlenir)
    return f"The answer is {answers[0]}."


print("Dataset loading functions ready.")

Dataset loading functions ready.


In [ ]:
# Tüm veri setlerini yükle
all_datasets = {}
for ds_key in DATASET_CONFIGS:
    print(f"\nLoading {ds_key}...")
    instances, passage_pool = load_and_prepare_dataset(ds_key)
    all_datasets[ds_key] = {
        "instances": instances,
        "passage_pool": passage_pool,
    }
    print(f"  {ds_key}: {len(instances)} instances, {len(passage_pool)} passages in pool")
    # Gold context coverage kontrolü
    has_real_gold = sum(1 for inst in instances if not inst["gold_context"].startswith("The answer is"))
    print(f"  Real gold context found: {has_real_gold}/{len(instances)}")

print("\nAll datasets loaded.")


Loading squad...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

...

rc.nocontext/test-00000-of-00001.parquet:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/138384 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

  triviaqa: 1000 instances, 50000 passages in pool
  Real gold context found: 478/1000

All datasets loaded.


---
## 4. Retriever Sınıfları

In [ ]:
class BM25Retriever:
    """Sparse retriever using BM25 (Okapi)."""

    def __init__(self, passages: List[str]):
        self.passages = passages
        tokenized = [p.lower().split() for p in passages]
        self.bm25 = BM25Okapi(tokenized)

    def retrieve(self, query: str, top_k: int = 1) -> Tuple[List[str], float]:
        sync_cuda()
        start = time.perf_counter()

        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]

        sync_cuda()
        end = time.perf_counter()

        retrieved = [self.passages[i] for i in top_indices]
        retrieval_ms = (end - start) * 1000.0
        return retrieved, retrieval_ms


class DenseRetriever:
    """Dense retriever using Sentence-BERT + FAISS."""

    def __init__(self, passages: List[str], model_name: str, device: str = DEVICE):
        self.passages = passages
        self.embedder = SentenceTransformer(model_name, device=device)

        # Encode passages
        print(f"  Encoding {len(passages)} passages with {model_name}...")
        embeddings = self.embedder.encode(
            passages,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings)

    def retrieve(self, query: str, top_k: int = 1) -> Tuple[List[str], float]:
        sync_cuda()
        start = time.perf_counter()

        query_emb = self.embedder.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True
        ).astype("float32")
        scores, indices = self.index.search(query_emb, k=top_k)

        sync_cuda()
        end = time.perf_counter()

        retrieved = [self.passages[int(idx)] for idx in indices[0]]
        retrieval_ms = (end - start) * 1000.0
        return retrieved, retrieval_ms


class HybridRetriever:
    """Hybrid retriever: alpha * dense_score + (1-alpha) * normalized_bm25_score."""

    def __init__(self, passages: List[str], model_name: str, alpha: float = 0.5, device: str = DEVICE):
        self.passages = passages
        self.alpha = alpha

        # BM25 component
        tokenized = [p.lower().split() for p in passages]
        self.bm25 = BM25Okapi(tokenized)

        # Dense component
        self.embedder = SentenceTransformer(model_name, device=device)
        print(f"  Encoding {len(passages)} passages for hybrid retriever...")
        self.passage_embeddings = self.embedder.encode(
            passages,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        dimension = self.passage_embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(self.passage_embeddings)

    def retrieve(self, query: str, top_k: int = 1) -> Tuple[List[str], float]:
        sync_cuda()
        start = time.perf_counter()

        # BM25 scores
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        # Min-max normalize BM25
        bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
        if bm25_max > bm25_min:
            bm25_norm = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
        else:
            bm25_norm = np.zeros_like(bm25_scores)

        # Dense scores
        query_emb = self.embedder.encode(
            [query], convert_to_numpy=True, normalize_embeddings=True
        ).astype("float32")
        dense_scores = (self.passage_embeddings @ query_emb.T).flatten()
        # Min-max normalize dense
        d_min, d_max = dense_scores.min(), dense_scores.max()
        if d_max > d_min:
            dense_norm = (dense_scores - d_min) / (d_max - d_min)
        else:
            dense_norm = np.zeros_like(dense_scores)

        # Combine
        combined = self.alpha * dense_norm + (1 - self.alpha) * bm25_norm
        top_indices = np.argsort(combined)[::-1][:top_k]

        sync_cuda()
        end = time.perf_counter()

        retrieved = [self.passages[i] for i in top_indices]
        retrieval_ms = (end - start) * 1000.0
        return retrieved, retrieval_ms


print("Retriever classes ready.")

Retriever classes ready.


---
## 5. Generator Yükleme ve Çıkarım Fonksiyonları

In [ ]:
class GeneratorWrapper:
    """Wrapper for loading and using a seq2seq generator."""

    def __init__(self, model_name: str, device: str = DEVICE):
        self.model_name = model_name
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Tuple[str, float]:
        inputs = self.tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=MAX_INPUT_LENGTH
        ).to(self.device)

        sync_cuda()
        start = time.perf_counter()

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
                early_stopping=True
            )

        sync_cuda()
        end = time.perf_counter()

        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        generation_ms = (end - start) * 1000.0
        return answer, generation_ms

    def free_memory(self):
        del self.model
        del self.tokenizer
        torch.cuda.empty_cache() if torch.cuda.is_available() else None


print("Generator wrapper ready.")

Generator wrapper ready.


---
## 6. İstatistiksel Analiz Fonksiyonları

In [ ]:
def compute_instance_f1(prediction: str, gold_answers: List[str]) -> float:
    """Tek bir instance için en yüksek F1 skorunu hesapla."""
    pred_tokens = normalize_text(prediction).split()
    if not pred_tokens:
        return 0.0

    best_f1 = 0.0
    for gold in gold_answers:
        gold_tokens = normalize_text(gold).split()
        if not gold_tokens:
            continue
        common = set(pred_tokens) & set(gold_tokens)
        num_common = sum(min(pred_tokens.count(w), gold_tokens.count(w)) for w in common)
        if num_common == 0:
            continue
        precision = num_common / len(pred_tokens)
        recall = num_common / len(gold_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        best_f1 = max(best_f1, f1)
    return best_f1


def compute_instance_em(prediction: str, gold_answers: List[str]) -> float:
    """Tek bir instance için EM (0 veya 1)."""
    norm_pred = normalize_text(prediction)
    return float(any(normalize_text(g) == norm_pred for g in gold_answers))


def bootstrap_ci(scores: np.ndarray, n_resamples: int = BOOTSTRAP_N_RESAMPLES,
                 ci_level: float = BOOTSTRAP_CI_LEVEL, seed: int = SEED) -> dict:
    """
    Bootstrap confidence interval for the mean.
    Returns: {mean, ci_lower, ci_upper, std}
    """
    rng = np.random.RandomState(seed)
    n = len(scores)
    boot_means = np.array([
        np.mean(rng.choice(scores, size=n, replace=True))
        for _ in range(n_resamples)
    ])

    alpha = 1 - ci_level
    ci_lower = np.percentile(boot_means, 100 * alpha / 2)
    ci_upper = np.percentile(boot_means, 100 * (1 - alpha / 2))

    return {
        "mean": float(np.mean(scores)),
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
        "std": float(np.std(boot_means)),
    }


def wilcoxon_test(scores_a: np.ndarray, scores_b: np.ndarray) -> dict:
    """
    Wilcoxon signed-rank test between two paired score arrays.
    Returns: {statistic, p_value, significant_at_005}
    """
    diff = scores_a - scores_b
    # Remove zero differences for Wilcoxon
    nonzero = diff[diff != 0]
    if len(nonzero) < 10:
        return {"statistic": None, "p_value": None, "significant_at_005": None}

    stat, p_val = stats.wilcoxon(nonzero)
    return {
        "statistic": float(stat),
        "p_value": float(p_val),
        "significant_at_005": bool(p_val < 0.05),
    }


print("Statistical analysis functions ready.")

Statistical analysis functions ready.


---
## 7. Ana Deney Döngüsü

Her (dataset, generator, retriever, top_k) kombinasyonu için:
1. Retrieved-context cevapları üret
2. Gold-context cevapları üret
3. Instance-level EM ve F1 hesapla
4. Bootstrap CI ve Wilcoxon testi uygula

In [ ]:
@dataclass
class ExperimentResult:
    dataset: str
    generator: str
    retriever: str
    top_k: int
    n_instances: int
    retrieval_accuracy: float
    # RAG metrics + CI
    rag_em_mean: float
    rag_em_ci_lower: float
    rag_em_ci_upper: float
    rag_f1_mean: float
    rag_f1_ci_lower: float
    rag_f1_ci_upper: float
    # Gold metrics + CI
    gold_em_mean: float
    gold_em_ci_lower: float
    gold_em_ci_upper: float
    gold_f1_mean: float
    gold_f1_ci_lower: float
    gold_f1_ci_upper: float
    # Deltas
    delta_em: float
    delta_f1: float
    # Wilcoxon
    wilcoxon_f1_p: Optional[float]
    wilcoxon_f1_sig: Optional[bool]
    # Subset analysis
    correct_retrieval_n: int
    correct_retrieval_rag_f1: float
    incorrect_retrieval_n: int
    incorrect_retrieval_rag_f1: float
    # Latency
    avg_rag_latency_ms: float
    avg_gold_latency_ms: float
    avg_retrieval_ms: float
    # RBDF metric (Hakem Yorumu 5)
    rbdf_score: float  # Retrieval Bottleneck Diagnostic Score


print("ExperimentResult dataclass ready.")

ExperimentResult dataclass ready.


In [ ]:
def run_single_experiment(
    dataset_key: str,
    generator_key: str,
    retriever_key: str,
    top_k: int,
    generator: GeneratorWrapper,
    retriever_obj,
    instances: List[dict],
) -> ExperimentResult:
    """
    Tek bir (dataset, generator, retriever, top_k) kombinasyonu için deney çalıştır.
    """

    rag_f1_scores = []
    rag_em_scores = []
    gold_f1_scores = []
    gold_em_scores = []
    retrieval_correct_flags = []
    rag_latencies = []
    gold_latencies = []
    retrieval_latencies = []

    for inst in tqdm(instances, desc=f"{dataset_key}/{generator_key}/{retriever_key}/k={top_k}", leave=False):
        question = inst["question"]
        gold_context = inst["gold_context"]
        gold_answers = inst["gold_answers"]

        # --- Retrieved-context ---
        retrieved_passages, retrieval_ms = retriever_obj.retrieve(question, top_k=top_k)
        # top-k passage'ları birleştir
        combined_context = " ".join(retrieved_passages)
        rag_prompt = build_prompt(combined_context, question)
        rag_answer, rag_gen_ms = generator.generate(rag_prompt)
        rag_total_ms = retrieval_ms + rag_gen_ms

        # Retrieval doğruluğu: top-k passage'lardan herhangi biri cevap içeriyor mu?
        retrieval_correct = any(
            any_answer_in_context(gold_answers, p) for p in retrieved_passages
        )

        # --- Gold-context ---
        gold_prompt = build_prompt(gold_context, question)
        gold_answer, gold_gen_ms = generator.generate(gold_prompt)

        # Instance-level skorlar
        rag_f1 = compute_instance_f1(rag_answer, gold_answers)
        rag_em = compute_instance_em(rag_answer, gold_answers)
        gold_f1 = compute_instance_f1(gold_answer, gold_answers)
        gold_em = compute_instance_em(gold_answer, gold_answers)

        rag_f1_scores.append(rag_f1)
        rag_em_scores.append(rag_em)
        gold_f1_scores.append(gold_f1)
        gold_em_scores.append(gold_em)
        retrieval_correct_flags.append(retrieval_correct)
        rag_latencies.append(rag_total_ms)
        gold_latencies.append(gold_gen_ms)
        retrieval_latencies.append(retrieval_ms)

    # Convert to numpy
    rag_f1_arr = np.array(rag_f1_scores)
    rag_em_arr = np.array(rag_em_scores)
    gold_f1_arr = np.array(gold_f1_scores)
    gold_em_arr = np.array(gold_em_scores)
    ret_correct = np.array(retrieval_correct_flags)

    # Bootstrap CI
    rag_em_ci = bootstrap_ci(rag_em_arr * 100)
    rag_f1_ci = bootstrap_ci(rag_f1_arr * 100)
    gold_em_ci = bootstrap_ci(gold_em_arr * 100)
    gold_f1_ci = bootstrap_ci(gold_f1_arr * 100)

    # Wilcoxon test (F1)
    wilcoxon_result = wilcoxon_test(gold_f1_arr, rag_f1_arr)

    # Subset analysis
    correct_mask = ret_correct == True
    incorrect_mask = ret_correct == False
    correct_n = int(correct_mask.sum())
    incorrect_n = int(incorrect_mask.sum())

    correct_rag_f1 = float(np.mean(rag_f1_arr[correct_mask]) * 100) if correct_n > 0 else 0.0
    incorrect_rag_f1 = float(np.mean(rag_f1_arr[incorrect_mask]) * 100) if incorrect_n > 0 else 0.0

    retrieval_accuracy = float(correct_n / len(instances) * 100)

    # --- RBDF Score (Hakem Yorumu 5) ---
    # Retrieval Bottleneck Diagnostic Score
    # RBDF = 1 - (F1_correct_retrieval / F1_gold)
    # 0'a yakınsa: bottleneck retrieval'da, generator iyi
    # 1'e yakınsa: generator da sorunlu
    gold_f1_mean = gold_f1_ci["mean"]
    if gold_f1_mean > 0:
        generation_gap_ratio = 1.0 - (correct_rag_f1 / gold_f1_mean)
    else:
        generation_gap_ratio = 1.0
    rbdf_score = max(0.0, min(1.0, generation_gap_ratio))

    return ExperimentResult(
        dataset=dataset_key,
        generator=generator_key,
        retriever=retriever_key,
        top_k=top_k,
        n_instances=len(instances),
        retrieval_accuracy=retrieval_accuracy,
        rag_em_mean=rag_em_ci["mean"],
        rag_em_ci_lower=rag_em_ci["ci_lower"],
        rag_em_ci_upper=rag_em_ci["ci_upper"],
        rag_f1_mean=rag_f1_ci["mean"],
        rag_f1_ci_lower=rag_f1_ci["ci_lower"],
        rag_f1_ci_upper=rag_f1_ci["ci_upper"],
        gold_em_mean=gold_em_ci["mean"],
        gold_em_ci_lower=gold_em_ci["ci_lower"],
        gold_em_ci_upper=gold_em_ci["ci_upper"],
        gold_f1_mean=gold_f1_ci["mean"],
        gold_f1_ci_lower=gold_f1_ci["ci_lower"],
        gold_f1_ci_upper=gold_f1_ci["ci_upper"],
        delta_em=gold_em_ci["mean"] - rag_em_ci["mean"],
        delta_f1=gold_f1_ci["mean"] - rag_f1_ci["mean"],
        wilcoxon_f1_p=wilcoxon_result["p_value"],
        wilcoxon_f1_sig=wilcoxon_result["significant_at_005"],
        correct_retrieval_n=correct_n,
        correct_retrieval_rag_f1=correct_rag_f1,
        incorrect_retrieval_n=incorrect_n,
        incorrect_retrieval_rag_f1=incorrect_rag_f1,
        avg_rag_latency_ms=float(np.mean(rag_latencies)),
        avg_gold_latency_ms=float(np.mean(gold_latencies)),
        avg_retrieval_ms=float(np.mean(retrieval_latencies)),
        rbdf_score=rbdf_score,
    )


print("Experiment runner ready.")

Experiment runner ready.


---
## 8. Tüm Deneyleri Çalıştır

**Dikkat:** Bu hücre Colab Pro GPU ile ~4-6 saat sürebilir.

Toplam deney sayısı = 3 dataset × 3 generator × 3 retriever × 4 top-k = **108 deney**

Memory yönetimi için generator'lar sırayla yüklenir ve boşaltılır.

In [ ]:
all_results = []

for gen_key, gen_model in GENERATOR_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Loading generator: {gen_key} ({gen_model})")
    print(f"{'='*60}")

    generator = GeneratorWrapper(gen_model)

    for ds_key in DATASET_CONFIGS:
        ds_data = all_datasets[ds_key]
        instances = ds_data["instances"]
        passage_pool = ds_data["passage_pool"]

        print(f"\n  Dataset: {ds_key} ({len(instances)} instances, {len(passage_pool)} passages)")

        # Warm-up
        for i in range(min(WARMUP_RUNS, len(instances))):
            prompt = build_prompt(instances[i]["gold_context"], instances[i]["question"])
            _ = generator.generate(prompt)

        for ret_key, ret_config in RETRIEVER_CONFIGS.items():
            print(f"\n    Building retriever: {ret_key}")

            if ret_config["type"] == "sparse":
                retriever = BM25Retriever(passage_pool)
            elif ret_config["type"] == "dense":
                retriever = DenseRetriever(passage_pool, ret_config["model"])
            elif ret_config["type"] == "hybrid":
                retriever = HybridRetriever(passage_pool, ret_config["model"], ret_config["alpha"])

            for k in TOP_K_VALUES:
                print(f"      Running: {ds_key}/{gen_key}/{ret_key}/k={k}")
                result = run_single_experiment(
                    dataset_key=ds_key,
                    generator_key=gen_key,
                    retriever_key=ret_key,
                    top_k=k,
                    generator=generator,
                    retriever_obj=retriever,
                    instances=instances,
                )
                all_results.append(result)
                print(f"        RAG F1: {result.rag_f1_mean:.2f} [{result.rag_f1_ci_lower:.2f}, {result.rag_f1_ci_upper:.2f}]")
                print(f"        Gold F1: {result.gold_f1_mean:.2f} [{result.gold_f1_ci_lower:.2f}, {result.gold_f1_ci_upper:.2f}]")
                print(f"        Retrieval Acc: {result.retrieval_accuracy:.2f}%, RBDF: {result.rbdf_score:.4f}")
                print(f"        Wilcoxon p={result.wilcoxon_f1_p}")

            # Free retriever memory
            del retriever

    # Free generator memory
    generator.free_memory()
    del generator
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n\nAll experiments completed. Total: {len(all_results)} configurations.")


Loading generator: flan-t5-small (google/flan-t5-small)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

      Running: natural_questions/flan-t5-large/sbert/k=1


natural_questions/flan-t5-large/sbert/k=1:   0%|          | 0/1000 [00:00<?, ?it/s]

        RAG F1: 5.28 [4.12, 6.55]
        Gold F1: 47.39 [44.42, 50.33]
        Retrieval Acc: 7.10%, RBDF: 0.2247
        Wilcoxon p=1.9811135859292354e-84
      Running: natural_questions/flan-t5-large/sbert/k=3


natural_questions/flan-t5-large/sbert/k=3:   0%|          | 0/1000 [00:00<?, ?it/s]

---
## 9. Sonuçları Kaydet

In [ ]:
# Results DataFrame
results_df = pd.DataFrame([asdict(r) for r in all_results])

# Save
results_csv = os.path.join(OUTPUT_DIR, "all_experiment_results.csv")
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

# JSON summary
with open(os.path.join(OUTPUT_DIR, "all_results.json"), "w") as f:
    json.dump([asdict(r) for r in all_results], f, indent=2, ensure_ascii=False)

display(results_df.head(10))
print(f"\nTotal configurations: {len(results_df)}")

---
## 10. Tablo 1: Genel Performans Karşılaştırması (Tüm Dataset × Generator × Retriever)

Makaledeki Table 1'in genişletilmiş hali — Bootstrap CI ile birlikte.

In [ ]:
# Makale Tablosu 1: top-k=1 için tüm kombinasyonlar
table1 = results_df[results_df["top_k"] == 1][[
    "dataset", "generator", "retriever",
    "rag_f1_mean", "rag_f1_ci_lower", "rag_f1_ci_upper",
    "gold_f1_mean", "gold_f1_ci_lower", "gold_f1_ci_upper",
    "delta_f1", "retrieval_accuracy",
    "wilcoxon_f1_p", "wilcoxon_f1_sig",
    "rbdf_score"
]].copy()

# Güzel formatlı CI sütunları
table1["RAG F1"] = table1.apply(
    lambda r: f"{r['rag_f1_mean']:.2f} [{r['rag_f1_ci_lower']:.2f}, {r['rag_f1_ci_upper']:.2f}]", axis=1)
table1["Gold F1"] = table1.apply(
    lambda r: f"{r['gold_f1_mean']:.2f} [{r['gold_f1_ci_lower']:.2f}, {r['gold_f1_ci_upper']:.2f}]", axis=1)
table1["Δ F1"] = table1["delta_f1"].apply(lambda x: f"{x:+.2f}")
table1["Ret. Acc."] = table1["retrieval_accuracy"].apply(lambda x: f"{x:.1f}%")
table1["p-value"] = table1["wilcoxon_f1_p"].apply(lambda x: f"{x:.2e}" if x else "N/A")
table1["RBDF"] = table1["rbdf_score"].apply(lambda x: f"{x:.4f}")

display_cols = ["dataset", "generator", "retriever", "RAG F1", "Gold F1", "Δ F1",
                "Ret. Acc.", "p-value", "RBDF"]

print("Table 1: Overall Performance (top-k=1)")
display(table1[display_cols])

table1[display_cols].to_csv(os.path.join(OUTPUT_DIR, "table1_overall.csv"), index=False)

---
## 11. Tablo 2: Top-k Ablation

In [ ]:
# SQuAD + flan-t5-base üzerinden top-k ablation
table2 = results_df[
    (results_df["dataset"] == "squad") &
    (results_df["generator"] == "flan-t5-base")
][[
    "retriever", "top_k",
    "rag_f1_mean", "rag_f1_ci_lower", "rag_f1_ci_upper",
    "retrieval_accuracy", "delta_f1", "rbdf_score"
]].copy()

table2["RAG F1"] = table2.apply(
    lambda r: f"{r['rag_f1_mean']:.2f} [{r['rag_f1_ci_lower']:.2f}, {r['rag_f1_ci_upper']:.2f}]", axis=1)

print("Table 2: Top-k Ablation (SQuAD, Flan-T5-Base)")
display(table2[["retriever", "top_k", "RAG F1", "retrieval_accuracy", "delta_f1", "rbdf_score"]])

table2.to_csv(os.path.join(OUTPUT_DIR, "table2_topk_ablation.csv"), index=False)

---
## 12. Tablo 3: Retrieval Bottleneck Decomposition (Genişletilmiş)

In [ ]:
# Tüm dataset-generator-retriever kombinasyonları için doğru/yanlış retrieval analizi
table3 = results_df[results_df["top_k"] == 1][[
    "dataset", "generator", "retriever",
    "correct_retrieval_n", "correct_retrieval_rag_f1",
    "incorrect_retrieval_n", "incorrect_retrieval_rag_f1",
    "gold_f1_mean", "rbdf_score"
]].copy()

print("Table 3: Retrieval Bottleneck Decomposition (top-k=1)")
display(table3)

---
## 13. Görselleştirmeler

In [ ]:
# Figure 1: Dataset × Generator F1 karşılaştırması (SBERT, k=1)
fig1_data = results_df[
    (results_df["retriever"] == "sbert") & (results_df["top_k"] == 1)
].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, ds_key in zip(axes, DATASET_CONFIGS.keys()):
    subset = fig1_data[fig1_data["dataset"] == ds_key]
    x = np.arange(len(subset))
    width = 0.35

    bars1 = ax.bar(x - width/2, subset["rag_f1_mean"], width, label="RAG",
                   yerr=[(subset["rag_f1_mean"] - subset["rag_f1_ci_lower"]).values,
                         (subset["rag_f1_ci_upper"] - subset["rag_f1_mean"]).values],
                   capsize=4, color="#4C72B0", alpha=0.85)
    bars2 = ax.bar(x + width/2, subset["gold_f1_mean"], width, label="Gold",
                   yerr=[(subset["gold_f1_mean"] - subset["gold_f1_ci_lower"]).values,
                         (subset["gold_f1_ci_upper"] - subset["gold_f1_mean"]).values],
                   capsize=4, color="#DD8452", alpha=0.85)

    ax.set_title(ds_key.upper(), fontsize=13, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(subset["generator"].values, rotation=15)
    ax.set_ylabel("F1 Score (%)" if ax == axes[0] else "")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_ylim(0, 100)

plt.suptitle("Figure 1: RAG vs Gold F1 by Dataset and Generator (SBERT, k=1)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig1_dataset_generator_f1.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: Top-k ablation (SQuAD, flan-t5-base, tüm retriever'lar)
fig2_data = results_df[
    (results_df["dataset"] == "squad") & (results_df["generator"] == "flan-t5-base")
].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ret_key in RETRIEVER_CONFIGS:
    subset = fig2_data[fig2_data["retriever"] == ret_key].sort_values("top_k")
    ax1.plot(subset["top_k"], subset["rag_f1_mean"], marker="o", label=ret_key, linewidth=2)
    ax1.fill_between(subset["top_k"], subset["rag_f1_ci_lower"], subset["rag_f1_ci_upper"], alpha=0.15)

    ax2.plot(subset["top_k"], subset["retrieval_accuracy"], marker="s", label=ret_key, linewidth=2)

# Gold baseline
gold_f1 = fig2_data["gold_f1_mean"].iloc[0]
ax1.axhline(y=gold_f1, color="gray", linestyle="--", label="Gold F1", alpha=0.7)

ax1.set_xlabel("Top-k")
ax1.set_ylabel("RAG F1 (%)")
ax1.set_title("RAG F1 vs Top-k")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(TOP_K_VALUES)

ax2.set_xlabel("Top-k")
ax2.set_ylabel("Retrieval Accuracy (%)")
ax2.set_title("Retrieval Accuracy vs Top-k")
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xticks(TOP_K_VALUES)

plt.suptitle("Figure 2: Top-k Ablation (SQuAD, Flan-T5-Base)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig2_topk_ablation.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: RBDF Heatmap — Retrieval Bottleneck Diagnostic Framework
fig3_data = results_df[results_df["top_k"] == 1].copy()
fig3_data["config"] = fig3_data["generator"] + " + " + fig3_data["retriever"]

pivot = fig3_data.pivot_table(index="config", columns="dataset", values="rbdf_score")

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn_r", ax=ax,
            vmin=0, vmax=0.5, linewidths=0.5,
            cbar_kws={"label": "RBDF Score (lower = retrieval is bottleneck)"})
ax.set_title("Figure 3: Retrieval Bottleneck Diagnostic Score (RBDF)\n"
             "Low RBDF → Generator is effective, retrieval is the bottleneck",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Generator + Retriever")
ax.set_xlabel("Dataset")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3_rbdf_heatmap.png"), dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4: Latency comparison
fig4_data = results_df[
    (results_df["top_k"] == 1) & (results_df["dataset"] == "squad")
].copy()

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(fig4_data))
width = 0.3

ax.bar(x - width, fig4_data["avg_retrieval_ms"], width, label="Retrieval", color="#4C72B0")
ax.bar(x, fig4_data["avg_rag_latency_ms"] - fig4_data["avg_retrieval_ms"], width,
       label="RAG Generation", color="#DD8452")
ax.bar(x + width, fig4_data["avg_gold_latency_ms"], width, label="Gold Generation", color="#55A868")

labels = fig4_data.apply(lambda r: f"{r['generator']}\n{r['retriever']}", axis=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Latency (ms)")
ax.set_title("Figure 4: Latency Decomposition (SQuAD, k=1)", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4_latency.png"), dpi=200, bbox_inches="tight")
plt.show()

---
## 14. Retrieval Bottleneck Diagnostic Framework — RBDF

Hakemin önerdiği pratik katkı: deneysel bulgulardan çıkan formalize bir teşhis çerçevesi.

### RBDF Tanımı

$$\text{RBDF} = 1 - \frac{F1_{\text{correct\_retrieval}}}{F1_{\text{gold}}}$$

- **RBDF ≈ 0**: Generator, doğru passage verildiğinde gold seviyesine yakın performans gösterir → **bottleneck retrieval'dadır**
- **RBDF → 1**: Generator, doğru passage verilse bile iyi performans gösteremez → **bottleneck generator'dadır**

### Ek Teşhis Metrikleri

| Metrik | Formül | Yorum |
|--------|--------|-------|
| Retrieval Accuracy (RA) | correct / total | Retriever'ın cevap içeren passage bulma oranı |
| Generation Gap Ratio (RBDF) | 1 - F1_correct/F1_gold | Generator'un context kullanım etkinliği |
| Retrieval Impact (RI) | F1_correct - F1_incorrect | Retrieval başarısının etkisi |
| Overall Bottleneck Attribution | RA × RBDF : low → retrieval bottleneck | Birleşik teşhis skoru |

In [ ]:
# RBDF Framework summary table
rbdf_summary = results_df[results_df["top_k"] == 1][[
    "dataset", "generator", "retriever",
    "retrieval_accuracy", "rbdf_score",
    "correct_retrieval_rag_f1", "incorrect_retrieval_rag_f1",
    "gold_f1_mean"
]].copy()

rbdf_summary["retrieval_impact"] = (
    rbdf_summary["correct_retrieval_rag_f1"] - rbdf_summary["incorrect_retrieval_rag_f1"]
)

rbdf_summary["bottleneck"] = rbdf_summary.apply(
    lambda r: "RETRIEVAL" if r["rbdf_score"] < 0.05 and r["retrieval_accuracy"] < 90
    else ("GENERATOR" if r["rbdf_score"] > 0.15 else "MIXED"),
    axis=1
)

print("RBDF Diagnostic Summary:")
display(rbdf_summary)

# Bottleneck distribution
print("\nBottleneck Distribution:")
print(rbdf_summary["bottleneck"].value_counts())

rbdf_summary.to_csv(os.path.join(OUTPUT_DIR, "rbdf_diagnostic_summary.csv"), index=False)

---
## 15. Tüm Çıktıları İndir

In [ ]:
print("Generated files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")

In [ ]:
# Google Colab download
try:
    from google.colab import files
    import shutil

    shutil.make_archive("experiment_results_extended", "zip", OUTPUT_DIR)
    files.download("experiment_results_extended.zip")
except ImportError:
    print("Not running in Colab. Files are in:", OUTPUT_DIR)